# V6.5-16 — Data v2 + YOLO26m teacher → distill YOLO26n (Raspberry Pi)
Mục tiêu: **mAP50 > 0.82** trên `val_clean`, model chạy được trên Raspberry Pi 3/5 (CPU, NCNN, không real-time).

- **Dataset** `datasets/v6p5_5cls_v2` (5 class, bỏ insulator): sửa nhãn bị lật ngang, bỏ box FO là người/cây/vùng trống, đưa lại box broken_strand thật đã review. Chi tiết: `data/label_audit/README.md`.
- **Val**: `val` = VAL gốc (frozen, so được với run cũ); `val_clean` = đã sửa nhãn → thước đo chính.
- **Thứ tự**: cell 3 teacher YOLO26m → cell 4 student YOLO26n distill (`distill_model`) → cell 5 export NCNN → cell 6 eval.
- Mỗi cell train **tự resume** từ `last.pt` nếu bị ngắt, và bỏ qua nếu đã xong → bị ngắt thì chạy lại đúng cell đó.
- **Watchdog**: 15 phút không có batch nào chạy xong (treo) → tự ngắt và resume; lỗi giữa chừng → tự thử lại (tối đa 3 lần); `last.pt` hỏng → lùi về `epochN.pt` (lưu mỗi 5 epoch). Tự bấm Interrupt thì dừng hẳn.
- Cell 3b: so với V6.4-clean YOLO11m@1024 (baseline cũ) trên cùng val gốc + val_clean.
- Khi train, log ultralytics được tắt (tránh output treo); mỗi epoch in **1 dòng**. Cell 7 xem tiến độ từ `results.csv` (dùng khi mở lại notebook).
- NCNN làm mAP50 giảm ~1.3 điểm (đo trên model v1) → student `.pt` cần ≥ ~0.83.

In [1]:
# 1. CONFIG + GATE (không train)
from pathlib import Path
import os, csv, time, logging
import torch, yaml, ultralytics
from ultralytics import YOLO
from ultralytics.utils import LOGGER

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next(p for p in candidates if (p / "pyproject.toml").is_file() and (p / "legacy/notebooks/v6p5").is_dir())
WEIGHTS = PROJECT / "legacy/notebooks/v6p5/weights"
DATASET = PROJECT.parent / "datasets/v6p5_5cls_v2"
DATA_YAML, VAL_CLEAN_YAML = DATASET / "data.yaml", DATASET / "data_val_clean.yaml"
OUT = PROJECT / "artifacts/v6p5_v2"
RUNS = OUT / "runs"
NAMES = ["insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand"]
TARGET_MAP50 = 0.82
IMGSZ, DEVICE, SEED = 640, 0, 20261011
WORKERS = min(8, os.cpu_count() or 4)
os.environ.setdefault("PYTHONHASHSEED", str(SEED))

# Công thức chung cho teacher và student (AdamW + cosine; plateau ở v6p5-14 giảm LR quá muộn)
COMMON = dict(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE, workers=WORKERS, seed=SEED, deterministic=True,
              optimizer="AdamW", lr0=1e-3, lrf=0.05, cos_lr=True, weight_decay=1e-3, warmup_epochs=3.0,
              close_mosaic=15, cache="disk", amp=True, pretrained=True, plots=False, save_period=5)

if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU — chuyển studio sang GPU rồi chạy lại.")
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(f"Cần ultralytics==8.4.172 (có distill_model); hiện tại={ultralytics.__version__}")
import ncnn  # export cho Raspberry Pi; thiếu thì: pip install ncnn pnnx

cfg = yaml.safe_load(DATA_YAML.read_text())
assert [cfg["names"][k] for k in sorted(cfg["names"])] == NAMES
for split in ["images/train", "images/val", "images/val_clean"]:
    print(split, sum(1 for _ in (DATASET / split).iterdir()))
for w in ["yolo26m.pt", "yolo26n.pt"]:
    assert (WEIGHTS / w).is_file(), w
OUT.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0), "| Ultralytics:", ultralytics.__version__, "| workers:", WORKERS)

images/train 17903
images/val 2272
images/val_clean 2271
GPU: NVIDIA L4 | Ultralytics: 8.4.172 | workers: 8


In [2]:
# 2. HÀM TRAIN: resume / bỏ qua nếu xong / watchdog chống treo / tự thử lại, in 1 dòng mỗi epoch
import gc, shutil, signal, threading
def _epoch_line(trainer):
    m, loss = trainer.metrics, trainer.label_loss_items(trainer.tloss)
    # sau epoch cuối ultralytics val lại bằng best.pt và gọi lại callback này
    final = trainer.stop and getattr(trainer, "_stop_seen", False)
    trainer._stop_seen = trainer.stop
    tag = "best.pt (val cuối)" if final else f"ep {trainer.epoch + 1:>3}/{trainer.epochs}"
    print(f"{trainer.args.name} {tag}  "
          f"box {float(loss['train/box_loss']):.3f} cls {float(loss['train/cls_loss']):.3f}  "
          f"mAP50 {m.get('metrics/mAP50(B)', 0):.4f}  "
          f"mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}  lr {trainer.optimizer.param_groups[0]['lr']:.2e}  "
          f"{time.strftime('%H:%M')}", flush=True)

HANG_MINUTES = 15  # không có batch nào chạy xong trong khoảng này → coi là treo
MAX_RETRIES = 3

class Watchdog:
    """Gửi SIGINT cho kernel nếu train không còn tiến triển (giống bấm Interrupt) để cell tự resume."""
    def __init__(self, minutes):
        self.limit, self.beat, self.armed, self.fired = minutes * 60, time.time(), False, False
        self._stop = threading.Event()
    def heartbeat(self, *_):  # gọi sau mỗi batch train/val
        self.beat, self.armed = time.time(), True
    def _run(self):
        while not self._stop.wait(30):
            if self.armed and time.time() - self.beat > self.limit:
                self.fired = True
                print(f"[watchdog] {self.limit // 60} phút không có batch nào chạy xong → ngắt để resume", flush=True)
                os.kill(os.getpid(), signal.SIGINT)
                return
    def __enter__(self):
        threading.Thread(target=self._run, daemon=True).start()
        return self
    def __exit__(self, *exc):
        self._stop.set()

def _latest_checkpoint(run):
    """last.pt, nếu hỏng (bị kill lúc đang ghi) thì lùi về epochN.pt mới nhất."""
    epochs = sorted(run.glob("weights/epoch*.pt"), key=lambda p: -int(p.stem[5:]))
    for p in [run / "weights/last.pt", *epochs]:
        if p.is_file():
            try:
                return p, torch.load(p, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[{run.name}] checkpoint hỏng, bỏ qua {p.name}: {e}")
    return None, None

def train_stage(name, model, **kw):
    run = RUNS / name
    best = run / "weights/best.pt"
    for attempt in range(1, MAX_RETRIES + 2):
        ckpt, ck = _latest_checkpoint(run)
        done = ck is not None and (ck.get("epoch", -1) == -1  # ultralytics đánh dấu run đã xong bằng epoch=-1
                                   or ck["epoch"] + 1 >= ck.get("train_args", {}).get("epochs", 10**9))
        if done and (ck.get("train_args", {}).get("epochs") == kw.get("epochs", ck.get("train_args", {}).get("epochs"))
                     or ck.get("epoch", -1) == -1):
            print(f"[{name}] đã xong, bỏ qua: {best}")
            return best
        if ck is not None and "epochs" in kw and ck.get("train_args", {}).get("epochs") != kw["epochs"]:
            print(f"[{name}] run cũ train {ck['train_args']['epochs']} epoch ≠ {kw['epochs']} → bắt đầu lại")
            ck = None
        if ck is not None:
            print(f"[{name}] resume từ {ckpt.name} (đã xong {ck['epoch'] + 1} epoch)")
            yolo, args = YOLO(str(ckpt)), dict(resume=True)
        else:
            if run.is_dir():  # không có checkpoint dùng được → làm lại từ đầu
                shutil.rmtree(run)
            print(f"[{name}] bắt đầu: {model}")
            yolo, args = YOLO(str(model)), dict(project=str(RUNS), name=name, exist_ok=False, **COMMON, **kw)
        wd = Watchdog(HANG_MINUTES)
        for event in ("on_train_batch_end", "on_val_batch_end", "on_fit_epoch_end"):
            yolo.add_callback(event, wd.heartbeat)
        yolo.add_callback("on_fit_epoch_end", _epoch_line)
        level = LOGGER.level
        LOGGER.setLevel(logging.WARNING)  # tắt progress bar + log chi tiết → output không bị treo
        try:
            with wd:
                yolo.train(**args)
            print(f"[{name}] xong: {best}")
            return best
        except KeyboardInterrupt:
            if not wd.fired:
                print(f"[{name}] dừng theo yêu cầu. Chạy lại cell này để resume.")
                raise
            print(f"[{name}] treo → resume (lần {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"[{name}] lỗi {type(e).__name__}: {e} → resume (lần {attempt}/{MAX_RETRIES})")
        finally:
            LOGGER.setLevel(level)
            del yolo
            gc.collect()
            torch.cuda.empty_cache()
        time.sleep(30)
    raise RuntimeError(f"[{name}] thất bại sau {MAX_RETRIES} lần thử lại — xem log ở trên")

In [3]:
# 3. TEACHER — YOLO26m (~6–8 giờ trên L4; epoch đầu chậm hơn vì tạo cache ảnh ~17 GB)
TEACHER_BEST = train_stage("teacher", WEIGHTS / "yolo26m.pt", epochs=80, batch=24, patience=20)

[teacher] bắt đầu: /teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/legacy/notebooks/v6p5/weights/yolo26m.pt
train: Caching images (45.9GB Disk): 100% ━━━━━━━━━━━━ 17903/17903 56.4it/s 5:170.0sss
val: Caching images (6.7GB Disk): 100% ━━━━━━━━━━━━ 2272/2272 158.8it/s 14.3s0.1s
teacher ep   1/80  box 2.266 cls 3.630  mAP50 0.2115  mAP50-95 0.0809  lr 3.33e-04  07:47
teacher ep   2/80  box 2.164 cls 2.727  mAP50 0.2207  mAP50-95 0.0879  lr 6.66e-04  07:53
teacher ep   3/80  box 2.125 cls 2.568  mAP50 0.3156  mAP50-95 0.1286  lr 9.98e-04  07:59
teacher ep   4/80  box 2.054 cls 2.315  mAP50 0.3877  mAP50-95 0.1596  lr 9.97e-04  08:05
teacher ep   5/80  box 2.023 cls 2.164  mAP50 0.3767  mAP50-95 0.1658  lr 9.94e-04  08:10
teacher ep   6/80  box 2.011 cls 2.084  mAP50 0.3990  mAP50-95 0.1769  lr 9.91e-04  08:16
teacher ep   7/80  box 1.968 cls 2.010  mAP50 0.4625  mAP50-95 0.1943  lr 9.87e-04  08:22
teacher ep   8/80  box 1.949 cls 1.927  mAP50 0.4528  mAP50-95 0.1922  lr 9.82e

In [ ]:
# 3b. SO SÁNH VỚI V6.4-clean YOLO11m@1024 (baseline cũ, tải từ Camber) — ~3 phút GPU
import pandas as pd

V64 = PROJECT / "artifacts/v6p4_clean"  # best.pt + val_clean_6cls/ (val_clean đánh lại id cho 6 class)
CMP = [  # (tên, weights, imgsz, data val gốc, data val_clean)
    ("V6.4 YOLO11m@1024", V64 / "best.pt", 1024, PROJECT.parent / "datasets/v6p5_roboflow_ablation/data.yaml",
     V64 / "val_clean_6cls/data.yaml"),
    ("Teacher YOLO26m@640", RUNS / "teacher/weights/best.pt", 640, DATA_YAML, VAL_CLEAN_YAML),
]
rows = []
for tag, w, sz, d_val, d_clean in CMP:
    for vname, vyaml in [("val", d_val), ("val_clean", d_clean)]:
        m = YOLO(str(w))
        b = m.val(data=str(vyaml), imgsz=sz, batch=8, device=DEVICE, plots=False, verbose=False,
                  project=str(OUT / "eval_cmp"), name=f"{tag.split()[0]}_{vname}", exist_ok=True).box
        for i, c in enumerate(b.ap_class_index):
            if m.names[int(c)] in NAMES:  # chỉ 5 class lỗi (bỏ insulator của model 6 class)
                rows.append({"model": tag, "val": vname, "class": m.names[int(c)], "P": float(b.p[i]),
                             "R": float(b.r[i]), "AP50": float(b.ap50[i]), "AP50_95": float(b.ap[i])})
cmp = pd.DataFrame(rows)
mean5 = cmp.groupby(["model", "val"], as_index=False)[["P", "R", "AP50", "AP50_95"]].mean().assign(**{"class": "MEAN_5"})
cmp = pd.concat([cmp, mean5], ignore_index=True)
cmp.to_csv(OUT / "compare_v6p4.csv", index=False)
display(cmp.pivot_table(index=["val", "class"], columns="model", values=["P", "R", "AP50", "AP50_95"]).round(3))

In [4]:
# 4. STUDENT — YOLO26n distill từ teacher (~5,5 giờ trên L4: ~5,4 phút/epoch vì chạy thêm teacher mỗi batch)
TEACHER_BEST = RUNS / "teacher/weights/best.pt"
assert TEACHER_BEST.is_file(), "Chạy cell 3 (teacher) trước"
STUDENT_BEST = train_stage("student_distill", WEIGHTS / "yolo26n.pt", epochs=60, batch=32, patience=15,
                           distill_model=str(TEACHER_BEST))

[student_distill] bắt đầu: /teamspace/studios/this_studio/Uav-Powerline-Defect-Detection/legacy/notebooks/v6p5/weights/yolo26n.pt
train: Caching images (45.9GB Disk): 100% ━━━━━━━━━━━━ 17903/17903 3.4Kit/s 5.2s0.2s
val: Caching images (6.7GB Disk): 100% ━━━━━━━━━━━━ 2272/2272 6.7Kit/s 0.3s0.1s
student_distill ep   1/120  box 2.167 cls 4.007  mAP50 0.1240  mAP50-95 0.0438  lr 3.33e-04  15:45
student_distill ep   2/120  box 2.021 cls 2.971  mAP50 0.2566  mAP50-95 0.1154  lr 6.66e-04  15:50
student_distill ep   3/120  box 1.987 cls 2.666  mAP50 0.2980  mAP50-95 0.1336  lr 9.99e-04  15:55
student_distill ep   4/120  box 1.961 cls 2.494  mAP50 0.3465  mAP50-95 0.1424  lr 9.99e-04  16:01
student_distill ep   5/120  box 1.942 cls 2.380  mAP50 0.4144  mAP50-95 0.1738  lr 9.97e-04  16:06
student_distill ep   6/120  box 1.939 cls 2.277  mAP50 0.4169  mAP50-95 0.1901  lr 9.96e-04  16:12
student_distill ep   7/120  box 1.918 cls 2.176  mAP50 0.4485  mAP50-95 0.1972  lr 9.94e-04  16:17
student_dist

KeyboardInterrupt: 

In [ ]:
# 5. EXPORT NCNN cho Raspberry Pi
STUDENT_BEST = RUNS / "student_distill/weights/best.pt"
NCNN_DIR = STUDENT_BEST.with_name("best_ncnn_model")
if not NCNN_DIR.is_dir():
    YOLO(str(STUDENT_BEST)).export(format="ncnn", imgsz=IMGSZ, half=False)
print("Copy thư mục này sang Pi:", NCNN_DIR)
print("Trên Pi: pip install ultralytics ncnn; YOLO('best_ncnn_model', task='detect').predict('anh.jpg', imgsz=640)")

In [ ]:
# 6. EVAL trên val gốc + val_clean, kiểm tra mục tiêu
import pandas as pd

MODELS = {
    "teacher_y26m": RUNS / "teacher/weights/best.pt",
    "student_distill": RUNS / "student_distill/weights/best.pt",
    "student_distill_ncnn": RUNS / "student_distill/weights/best_ncnn_model",  # bản chạy trên Pi (CPU)
    "y26n_v1_5clsclean": PROJECT / "artifacts/v6p5_train_local/20261010T133406Z_y26n_5cls_plateau_wd001/runs/yolo26n_v6p5_5clsclean_plateau_wd001/weights/best.pt",
}
rows = []
for vname, vyaml in [("val", DATA_YAML), ("val_clean", VAL_CLEAN_YAML)]:
    for tag, w in MODELS.items():
        if not w.exists():
            continue
        ncnn_model = w.is_dir()
        b = YOLO(str(w), task="detect").val(data=str(vyaml), imgsz=IMGSZ, batch=1 if ncnn_model else 16,
                                            device="cpu" if ncnn_model else DEVICE, plots=False, verbose=False,
                                            project=str(OUT / "eval"), name=f"{tag}_{vname}", exist_ok=True).box
        rows.append({"model": tag, "val": vname, "class": "ALL", "P": float(b.mp), "R": float(b.mr), "AP50": float(b.map50), "AP50_95": float(b.map)})
        for i, (c, a50, a) in enumerate(zip(b.ap_class_index, b.ap50, b.ap)):
            rows.append({"model": tag, "val": vname, "class": NAMES[int(c)], "P": float(b.p[i]), "R": float(b.r[i]), "AP50": float(a50), "AP50_95": float(a)})
res = pd.DataFrame(rows)
res.to_csv(OUT / "eval.csv", index=False)
display(res.pivot_table(index=["val", "class"], columns="model", values=["P", "R", "AP50", "AP50_95"]).round(4))
pi = res.query("model == 'student_distill_ncnn' and val == 'val_clean' and `class` == 'ALL'")
if len(pi):
    v = float(pi.AP50.iloc[0])
    print(f"Model Raspberry Pi (NCNN) trên val_clean: mAP50 = {v:.4f} →",
          "ĐẠT" if v > TARGET_MAP50 else "CHƯA ĐẠT", f"mục tiêu {TARGET_MAP50}")

In [ ]:
# 7. XEM TIẾN ĐỘ từ results.csv (khi mở lại notebook / kernel đã restart; không ảnh hưởng train)
import datetime as dt
for stage in ["teacher", "student_distill"]:
    f = RUNS / stage / "results.csv"
    if not f.exists():
        continue
    rows_ = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f.open())]
    best = max(rows_, key=lambda r: float(r["metrics/mAP50(B)"]))
    print(f"== {stage}: {len(rows_)} epoch | best mAP50 {float(best['metrics/mAP50(B)']):.4f} (ep {best['epoch']}) "
          f"| cập nhật {dt.datetime.fromtimestamp(f.stat().st_mtime):%H:%M:%S}")
    for r in rows_[-10:]:
        print(f"  ep {r['epoch']:>3}  box {float(r['train/box_loss']):.3f}  mAP50 {float(r['metrics/mAP50(B)']):.4f}  "
              f"mAP50-95 {float(r['metrics/mAP50-95(B)']):.4f}  lr {float(r['lr/pg0']):.2e}")